### <span style="color:red">DC_FW Project: Libraries</span>

In [ ]:
'''''
## Test setup for Neural Network's experiment 
#
# [MHSY25] H. Maskan, Y.Hou, S.Sra, A. Yurtsever
% "Revisiting Frank-Wolfe for Structured Nonconvex Optimization"
% 39th Conference on Neural Information Processing Systems (NeurIPS 2025).
% 
% contact information: https://github.com/hoomyhh
'''''
# This block imports libraries

# Basic libraries:
import os
import torch
import torch.nn as nn
# import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt
import numpy as np

#===========================================================
# Classification needs:
from torchvision import transforms
from torchvision import models

# For specific datasets 
# from torchvision.datasets import MNIST
from torchvision.datasets import CIFAR10

import pickle
import time
import datetime
import random


### <span style="color:blue">GPU/CPU</span>

In [ ]:
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"
print(f"Using {device} device")

### <span style="color:blue">Reproducible setting</span>

In [ ]:
def reset_seed(public_seed = 126):
    random.seed(public_seed)
    np.random.seed(public_seed)
    torch.manual_seed(public_seed) 
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    

### <span style="color:blue">Dataset pre-processing</span>

In [ ]:
# local dataset path 
data_path = "./CIFAR10" 


# Data set pre-processing
# Warning: This is dataset-wise setting
transform = transforms.Compose([transforms.ToTensor(),
                                transforms.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261))# Only for CIFAR10 dataset
                               ])

# Load all or partial data
Load_ALL = True

if Load_ALL is True:
    print("Load all data")
    train_dataset = CIFAR10(data_path, train = True, download = False, transform = transform)
    val_dataset = CIFAR10(data_path, train = False, download = False, transform = transform)
else:
    print("Load partial data")
    num_train_samples = 1024 * 6
    num_test_samples = 1024
    CIFAR10_dataset = CIFAR10(root = data_path, train = True, download = False, transform = transform)
    train_indices = random.sample(range(len(CIFAR10_dataset)), num_train_samples)
    test_indices = random.sample(range(len(CIFAR10_dataset)), num_test_samples)

    train_dataset = Subset(CIFAR10_dataset, train_indices)
    val_dataset = Subset(CIFAR10_dataset, test_indices)

### <span style="color:blue">Network structure</span>

In [ ]:
class CIFAR10CNN(nn.Module):
    def __init__(self):
        super(CIFAR10CNN, self).__init__()
        self.conv_block1 = nn.Sequential(nn.Conv2d(3, 32, kernel_size = 3, stride = 1, padding = 1),
                                         nn.ReLU(),
                                         nn.BatchNorm2d(32),
                                         nn.Conv2d(32, 32, kernel_size = 3, stride = 1, padding = 1),
                                         nn.ReLU(),
                                         nn.BatchNorm2d(32),
                                         nn.MaxPool2d(kernel_size = 2, stride = 2, padding = 0),
                                         nn.Dropout(0.01))
        
        self.conv_block2 = nn.Sequential(nn.Conv2d(32, 64, kernel_size = 3, stride = 1, padding = 1),
                                         nn.ReLU(),
                                         nn.BatchNorm2d(64),
                                         nn.Conv2d(64, 64, kernel_size = 3, stride = 1, padding = 1),
                                         nn.ReLU(),
                                         nn.BatchNorm2d(64),
                                         nn.MaxPool2d(kernel_size = 2, stride = 2, padding = 0),
                                         nn.Dropout(0.01))
        
        self.fc_block = nn.Sequential(nn.Flatten(),
                                      nn.Linear(64 * 8 * 8, 512), # 512 if following layers are applied
                                      nn.ReLU(),
                                      nn.BatchNorm1d(512),
                                      nn.Dropout(0.01),
                                      nn.Linear(512, 1024),
                                      nn.ReLU(),
                                      nn.BatchNorm1d(1024),
                                      nn.Dropout(0.01),
                                      nn.Linear(1024, 10))

    def forward(self, x):
        x = self.conv_block1(x)
        x = self.conv_block2(x)
        x = self.fc_block(x)
        return x

### <span style="color:red">The proposed DC-FW algorithm</span>

In [ ]:
# DC_FW for ICML adjustment

class DC_FW_ICML(torch.optim.Optimizer):
    def __init__(self, params, alpha, tuning_c, inner_upper):
        assert alpha > 0.0, f"Invalid alpha: {alpha}, it should be positive constant"
        assert tuning_c > 0.0, f"Invalid tuning_c: {tuning_c}, it should be positive constant"
        
        # Main variables
        self.iteration_counter = 1             # Count the number of TOTAL iterations (=outer_loop_t). Reset until re-instantiating optimizer
        self.inner_loop_upper = inner_upper    # Inner loop index k. should be very large
   
        self.gap = []                # For saving gap list for each parameter group
        self.tolerance = []          # For saving tolerance list for each parameter group
        self.para_t = []             # Update after end of the inner loop (The new [W_t])
        
        self.init_W_t = True         # The first loop in the first iteration requires the initialisation
        
        # Auxiliary variables
        self.gap_warn = 0            # For dectecting negative gap
         
        # defaults (All hyper-parameters should be included in this "defaults")
        defaults = dict(alpha = alpha, 
                        tuning_c = tuning_c)
        super(DC_FW_ICML, self).__init__(params, defaults)
        
        
    # For recovering optimizer state when loads a saved model (Not necessary in this case)
    def __setstate__(self, state):
        super(DC_FW_ICML, self).__setstate__(state)
        
        
    # Step function (optimiser.step())
    def step(self, closure = None):
        # closure is used to re-calcualate loss in one step (Not necessary in this case)
        loss = None
        if closure is not None:
            loss = closure()
            
        # group includes para and hyper-para
        # We use global setting (not layer specific setting) which indicates only one group
        # e.g., if we set different learning rate for different layers, we have multiple groups
        # Hyper-parameters can be accessed here as normal variables
        # ----------------------------------------------------
        for group in self.param_groups: 

            # Extract and re-state hyper-parameters 
            alpha = group['alpha']
            tuning_c = group['tuning_c']

            # Initialise or update variables in the outer_loop
            # Initialisation
            if self.init_W_t is True: # Initialisation (only once per instantiation of optimiser)
                for para_index, para_tk in enumerate(group['params']):
                    self.para_t.append(para_tk.detach().clone())
                    G_tk = para_tk.grad.data
                    S_tk = -tuning_c * torch.sign(G_tk)
                    D_tk = S_tk - para_tk.data
                    
                    # To initialise gap and tolerance, method 1:
                    # %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                    # self.gap.append(torch.sum(-G_tk * D_tk))
                    # self.tolerance.append(torch.sum(-G_tk * D_tk) * 0.9)
                    # %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                    
                    # To initialise gap and tolerance, method 2:
                    # %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                    if G_tk.ndim == 2:
                        self.gap.append(torch.linalg.matrix_norm(G_tk, ord = 1) * torch.linalg.matrix_norm(D_tk, ord = np.inf))
                    else:
                        self.gap.append(torch.linalg.vector_norm(G_tk, ord = 1) * torch.linalg.vector_norm(D_tk, ord = np.inf))
                    self.tolerance.append(self.gap[para_index] * 0.9)
                    # %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                del para_index, para_tk, G_tk, S_tk, D_tk
                self.init_W_t = False # No more initialisations of the current optimiser
                
                print(f"Parameters is divided to {len(self.para_t)} (= #layers * 2 when bias available) small groups and update independently")
                print(f"Initialisation: self.gap {self.gap}\nTol is {self.tolerance}")
                print("Initialisation for the loops has done (only once)\n\n")
                
            # Update for outer_loop
            else:
                for para_index, para_tk in enumerate(group['params']):
                    G_tk = para_tk.grad.data
                    S_tk = -tuning_c * torch.sign(G_tk)
                    D_tk = S_tk - para_tk.data # (S_tk - W_tk), same as (S_tk - W_t)
                    
                    # To calculate gap in the outer loop, method 1:
                    # %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                    # Do: gap = - G_{t,k} dot D_{t_k} (Previously we use G_tk dot S_tk)
                    # self.gap[para_index] = torch.sum(-G_tk * D_tk)
                    # %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                    
                    # To calculate gap in the outer loop, method 2:
                    # %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                    # Do: gap = L1norm of G_tk * infnorm of D_tk 
                    if G_tk.ndim == 2:
                        self.gap[para_index] = torch.linalg.matrix_norm(G_tk, ord = 1) * torch.linalg.matrix_norm(D_tk, ord = np.inf)
                    else:
                        self.gap[para_index] = torch.linalg.vector_norm(G_tk, ord = 1) * torch.linalg.vector_norm(D_tk, ord = np.inf)
                    
                    if self.gap[para_index] < self.tolerance[para_index]:
                        self.tolerance[para_index] = self.gap[para_index] * 0.9
                    # %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                del para_index, para_tk, G_tk, S_tk, D_tk
                
            
            # Update the set of parameters (from shallow to deep, weights to bias)
            # ----------------------------------------------------
            for para_index, para_tk in enumerate(group['params']):
                # Check gradient and skip the parameter without gradient
                if para_tk.grad is None:
                    continue
                
                # Algorithm in inner_loop. Can be breaked by the gap or finished by reaching upper limit
                # ----------------------------------------------------
                for inner_counter in range (self.inner_loop_upper):
                    
                    # '''''''''''''''''''''''''''''''''''''''''''''''''''''''''
                    # Do: G_{t,k} = gradient_f(W_{t}) - alpha * (W_t - W_{t,k})
                    G_tk = para_tk.grad.data - alpha * (self.para_t[para_index] - para_tk.data)
                    # '''''''''''''''''''''''''''''''''''''''''''''''''''''''''
                    
                    # '''''''''''''''''''''''''''''''''''''''''''''''''''''''''
                    # Do: S_{t,k} = lmo_D (G_{t,k}), Where lmo_D(G_t) = -c*sign(G_t)  [If norm(W,inf) <= c]
                    S_tk = -tuning_c * torch.sign(G_tk)
                    # '''''''''''''''''''''''''''''''''''''''''''''''''''''''''
                    
                    # '''''''''''''''''''''''''''''''''''''''''''''''''''''''''
                    # Do: D_{t,k} = S_{t,k} - W{t,k}
                    D_tk = S_tk - para_tk.data
                    # '''''''''''''''''''''''''''''''''''''''''''''''''''''''''
                    
                    # '''''''''''''''''''''''''''''''''''''''''''''''''''''''''
                    # To calculate gap in the inner loop, method 1:
                    # %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                    # Do: gap = - G_{t,k} dot D_{t_k} (Previously we use G_tk dot S_tk)
                    # self.gap[para_index] = torch.sum(-G_tk * D_tk)
                    # %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                    
                    # To calculate gap in the inner loop, method 2:
                    # %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                    # Do: gap = L1_norm of G_tk * inf_norm of D_tk
                    if G_tk.ndim == 2:
                        self.gap[para_index] = torch.linalg.matrix_norm(G_tk, ord = 1) * torch.linalg.matrix_norm(D_tk, ord = np.inf)
                        D_tk_l2norm = torch.linalg.matrix_norm(D_tk, ord = 'fro') ** 2
                    else:
                        self.gap[para_index] = torch.linalg.vector_norm(G_tk, ord = 1) * torch.linalg.vector_norm(D_tk, ord = np.inf)
                        D_tk_l2norm = torch.linalg.vector_norm(D_tk, ord = 2) ** 2
                    # %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                    # '''''''''''''''''''''''''''''''''''''''''''''''''''''''''
                    
                    # Warning check: negative gap
                    if self.gap[para_index].item() < self.gap_warn:
                        print(f"Inner loop gap {self.gap[para_index]} (Threo: {self.gap_warn}) for para_index {para_index} in inner loop {inner_counter} at iteration {self.iteration_counter}")         
                    
                    # '''''''''''''''''''''''''''''''''''''''''''''''''''''''''
                    # Check gap condition
                    if self.gap[para_index].item() <= (0.5 * self.tolerance[para_index].item()):
                        # print(f"gap:{self.gap[para_index]} <= 0.5 * {self.tolerance[para_index]} (tolerance), in inner loop {inner_counter} with para index {para_index}")
                        break
                    else:
                        # Do: eta_{t,k} = 2/(s+1)
                        # eta_tk = 2.0 / (self.iteration_counter + 1) # eta_{t,k} = 2/(s+1)
                        # eta_tk = 2.0 / (inner_counter + 1)        # eta_{t,k} = 2/(k+1)
                        
                        # If line search is applied here, eta_tk should be in [0,1]
                        # eta_tk = np.minimum(np.maximum(self.gap[para_index].item() / (D_tk_l2norm * alpha), 0.0), 1.0)
                        eta_tk = torch.minimum(torch.maximum(self.gap[para_index] / (D_tk_l2norm * alpha), torch.tensor(0.0, device = self.gap[para_index].device)), torch.tensor(1.0, device = self.gap[para_index].device))
                        # Do: W_{t,k+1} = W_{t,k} + eta_{t,k} (S_{t,k} - W_{t,k})
                        para_tk.data = para_tk.data + eta_tk * (S_tk - para_tk.data)
                    # '''''''''''''''''''''''''''''''''''''''''''''''''''''''''
                
                # ====== Inner loop ends here ==============
                # ====== below outside of this loop ======== 
                
                # save the updated W_tk for each set of parameters after inner loop
                # Do: W_{t+1} = W{t,k}
                self.para_t[para_index] = para_tk.data.detach().clone()
                
                # Warning (Not necessary, for tuning): 
                # if inner_counter == (self.inner_loop_upper - 1):
                #     print("Reach the inner loop upper bound")
                
                
            # ====== loop for parameters (from shallow to deep layer, weights to bias) ======
            # ====== It loops (#layers * 2) times ===========================================
            # ====== below outside of this loop =============================================

            
        # ====== loop for group of parameters and hyper-parameters ======
        # ====== It loops (1) times since we use global setting    ======
        # ====== below outside of this loop =============================
        
        self.iteration_counter += 1 # Reset until re-instantiating optimizer 
        return loss
        

### <span style="color:purple">The classical FW algorithm</span>

In [ ]:
# FW for ICML adjustment

class FW_ICML(torch.optim.Optimizer):
    def __init__(self, params, tuning_c):
        assert tuning_c > 0.0, f"Invalid tuning_c: {tuning_c}, it should be positive constant"
        assert 'alpha' not in locals(), "Normal FW should NOT have alpha"
        self.iteration_counter = 1 

        defaults = dict(tuning_c = tuning_c)
        super(FW_ICML, self).__init__(params, defaults)
        
        
    def __setstate__(self, state):
        super(FW_ICML, self).__setstate__(state)
        
        
    def step(self, closure = None):
        loss = None
        if closure is not None:
            loss = closure()
            
        for group in self.param_groups:    
            tuning_c =  group['tuning_c']
            
            for para_index, para in enumerate(group['params']):
                if para.grad is None:
                    continue
                    
                # Do: G_{t} = gradient_f(W_{t})
                G_t = para.grad.data
                
                # Do: S_{t} = lmo_D (G_{t}), Where lmo_D(G_t) = -c*sign(G_t)
                S_t = -tuning_c * torch.sign(G_t)
                
                # Do: eta_{t} = 2/(s+1)
                eta_t = 2.0 / (self.iteration_counter + 1)
                
                # Do: W_{t+1} = W_{t} + eta_{t} * (S_{t} - W_{t})
                para.data = para.data + eta_t * (S_t - para.data)
                
        self.iteration_counter += 1
        return loss

### <span style="color:blue">Train / Validation framework</span>

In [ ]:
def testFrame(model, optimizer, loss_fn, train_loader, val_loader, num_epochs, device):
    model.to(device)
    train_loss_epoch, val_loss_epoch = [], []
    train_acc_epoch, val_acc_epoch = [], []

    for epoch in range(1, num_epochs + 1):
        model, train_loss, train_acc = training_loop(model,
                                                     optimizer,
                                                     loss_fn,
                                                     train_loader,
                                                     device)
        train_loss_epoch.append(train_loss)
        train_acc_epoch.append(train_acc)
        
        val_loss, val_acc = validation(model,
                                       loss_fn,
                                       val_loader,
                                       device)
        val_loss_epoch.append(val_loss)
        val_acc_epoch.append(val_acc)

        print(f"epoch {epoch}/{num_epochs} training_loss: {train_loss:.12f} val_loss: {val_loss:.12f} train_acc: {train_acc:.5f} val_acc: {val_acc:.5f}")

    return model, train_loss_epoch, val_loss_epoch, train_acc_epoch, val_acc_epoch


def training_loop(model, optimizer, loss_fn, train_loader, device):
    model.train()
    total_train_loss = 0
    train_acc_batches = 0

    for data, targets in train_loader:
        data = data.to(device)
        targets = targets.to(device)
        outputs = model(data)
        loss = loss_fn(outputs, targets)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()
        train_acc_batches += (outputs.argmax(1) == targets).sum().item()

    avg_train_loss = total_train_loss / len(train_loader)
    train_accuracy = train_acc_batches / len(train_loader.dataset)

    return model, avg_train_loss, train_accuracy


def validation(model, loss_fn, val_loader, device):
    model.eval()
    total_val_loss = 0
    val_acc_batches = 0

    with torch.no_grad():
        for data, targets in val_loader:
            data = data.to(device)
            targets = targets.to(device)
            outputs = model(data)
            val_batch_loss = loss_fn(outputs, targets)

            total_val_loss += val_batch_loss.item()
            val_acc_batches += (outputs.argmax(1) == targets).sum().item()

    avg_val_loss = total_val_loss / len(val_loader)
    val_accuracy = val_acc_batches / len(val_loader.dataset)

    return avg_val_loss, val_accuracy


### <span style="color:blue">Helper Functions</span>

In [ ]:
# use to store loss or accuracy
def store_metrics(a, b):
    if len(a) == 0:
        a = [b]
    else:
        a.append(b)
    return a

# Averaging results among various seeds
def take_avg(list_in):
    if len(list_in) == 0:
        list_out = list_in
    else:    
        sum_list = list_in[0]
        if len(list_in) > 1: 
            for i in range (1, len(list_in)):
                sum_list = list(map(lambda x, y: x + y, sum_list, list_in[i]))
        list_out = [x / len(list_in) for x in sum_list]
    
    return list_out

# Averaging results of last few epochs
def avg_stable(numbers, last_n):
    if not numbers:
        raise ValueError("Void list")
    avg_last = numbers[-last_n:]

    return sum(avg_last) / len(avg_last)


### <span style="color:red">Experiment setting</span>

In [ ]:
reset_seed()

batch_size = 256
num_workers = 2
if batch_size == 'full': # Test for GD 
    train_dataloader = DataLoader(train_dataset, batch_size = len(train_dataset), shuffle = True, num_workers = num_workers)
    val_dataloader = DataLoader(val_dataset, batch_size = len(val_dataset), shuffle = False, num_workers = num_workers)
else:
    train_dataloader = DataLoader(train_dataset, batch_size = batch_size, shuffle = True, num_workers = num_workers)
    val_dataloader = DataLoader(val_dataset, batch_size = batch_size, shuffle = False, num_workers = num_workers)


num_epochs = 100
num_seeds = 1
last_n = 5

loss_fn = nn.CrossEntropyLoss()
time_record = []

# DC_FW configuration and
#----------------------
alpha = 1             # DC_FW only  (1, 10, 100, 500, 1000)
inner_upper = 10000   # DC_FW only (fixed)
tuning_c = 1          # DC_FW and FW shared (1, 10, 100)
#----------------------

# Set result path
result_path = './CIFAR10_CNN_Alpha_{0}_c_{1}_innerup_{2}_BS_{3}_E_{4}_S_{5}'.format(alpha, tuning_c, inner_upper, batch_size, num_epochs, num_seeds)

if not os.path.exists(result_path):
    # If the folder doesn't exist, create it
    os.makedirs(result_path)
    print("Result folder created successfully.")       
else:
    print("Result Result folder already exists.")
    
reset_seed()
model_seed = random.sample(range(1, 1024), num_seeds) # "Too many seeds"


### <span style="color:blue">Model_0: DC_FW</span>

In [ ]:
full_train_loss_DC_FW = []
full_val_loss_DC_FW = []
full_train_acc_DC_FW = []
full_val_acc_DC_FW = []

start_time = time.time()
time_record.append(start_time)

for i in range(num_seeds): # Number of loops indicates how many seeds are averaged
    reset_seed(public_seed = model_seed[i])
    print(f"Training model with DC_FW in {i + 1}th seed, model is initialised by the seed: {model_seed[i]}\n")
    
    model = CIFAR10CNN()
    optimizer = DC_FW_ICML(model.parameters(), alpha = alpha, tuning_c = tuning_c, inner_upper = inner_upper)
    model, train_losses, val_losses, train_acc_e, val_acc_e = testFrame(model = model, 
                                                                        optimizer = optimizer, 
                                                                        loss_fn = loss_fn, 
                                                                        train_loader = train_dataloader, 
                                                                        val_loader = val_dataloader, 
                                                                        num_epochs = num_epochs, 
                                                                        device = device)    
    
    full_train_loss_DC_FW = store_metrics(full_train_loss_DC_FW, train_losses)
    full_val_loss_DC_FW = store_metrics(full_val_loss_DC_FW, val_losses)
    full_train_acc_DC_FW = store_metrics(full_train_acc_DC_FW, train_acc_e)
    full_val_acc_DC_FW = store_metrics(full_val_acc_DC_FW, val_acc_e)
    
avg_train_loss_DC_FW = take_avg(full_train_loss_DC_FW)
avg_val_loss_DC_FW = take_avg(full_val_loss_DC_FW)
avg_train_acc_DC_FW = take_avg(full_train_acc_DC_FW)
avg_val_acc_DC_FW = take_avg(full_val_acc_DC_FW)

final_train_loss_DC_FW = avg_stable(avg_train_loss_DC_FW, last_n = last_n)
final_val_loss_DC_FW = avg_stable(avg_val_loss_DC_FW, last_n = last_n)
final_train_acc_DC_FW = avg_stable(avg_train_acc_DC_FW, last_n = last_n)
final_val_acc_DC_FW = avg_stable(avg_val_acc_DC_FW, last_n = last_n)

end_time = time.time()    
time_record.append(end_time)
duration = end_time - start_time
time_record.append(duration)

del model # Can be commented 

### <span style="color:blue">Model_1: FW</span>

In [ ]:
full_train_loss_FW = []
full_val_loss_FW = []
full_train_acc_FW = []
full_val_acc_FW = []

start_time = time.time()
time_record.append(start_time)

for i in range(num_seeds):
    reset_seed(public_seed = model_seed[i])
    print(f"Training model with FW in {i + 1}th seed, model is initialised by the seed: {model_seed[i]}\n")
    
    model = CIFAR10CNN()
    optimizer = FW_ICML(model.parameters(), tuning_c = tuning_c)
    model, train_losses, val_losses, train_acc_e, val_acc_e = testFrame(model = model, 
                                                                        optimizer = optimizer, 
                                                                        loss_fn = loss_fn, 
                                                                        train_loader = train_dataloader, 
                                                                        val_loader = val_dataloader, 
                                                                        num_epochs = num_epochs, 
                                                                        device = device)    
    full_train_loss_FW = store_metrics(full_train_loss_FW, train_losses)
    full_val_loss_FW = store_metrics(full_val_loss_FW, val_losses)
    full_train_acc_FW = store_metrics(full_train_acc_FW, train_acc_e)
    full_val_acc_FW = store_metrics(full_val_acc_FW, val_acc_e)
    
avg_train_loss_FW = take_avg(full_train_loss_FW)
avg_val_loss_FW = take_avg(full_val_loss_FW)
avg_train_acc_FW = take_avg(full_train_acc_FW)
avg_val_acc_FW = take_avg(full_val_acc_FW)

final_train_loss_FW = avg_stable(avg_train_loss_FW, last_n = last_n)
final_val_loss_FW = avg_stable(avg_val_loss_FW, last_n = last_n)
final_train_acc_FW = avg_stable(avg_train_acc_FW, last_n = last_n)
final_val_acc_FW = avg_stable(avg_val_acc_FW, last_n = last_n)

end_time = time.time()    
time_record.append(end_time)
duration = end_time - start_time
time_record.append(duration)

del model # Can be commented 

In [ ]:
print("all done")